# 3. Does a modern model fail the same way?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SanjanaJairam/abusive-language-detection/blob/main/notebooks/03_modern_model.ipynb)

Notebooks 1 and 2 audit models built with 2017-era methods. A fair question is whether a current transformer model has the same problems. This notebook runs two RoBERTa models from Cardiff NLP through the **same audits**:

- [`twitter-roberta-base-hate-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-hate-latest): hate or not hate, trained on 13 hate-speech datasets (Antypas & Camacho-Collados, 2023).
- [`twitter-roberta-base-offensive`](https://huggingface.co/cardiffnlp/twitter-roberta-base-offensive): offensive or not, trained on OLID through TweetEval (Barbieri et al., 2020).

Together they give the same three labels as before: **hate** if the hate model says so, otherwise **offensive** if the offensive model says so, otherwise **neither**.

**Read the results with one caveat.** The hate model's training data includes Davidson et al. (2017), listed on its model card as "Automated Hate Speech Detection", and OLID. So the Davidson and dialect results below are on tweets it has **already seen**: a best case for it, not a fair test. HateCheck and the identity sentences are new to it, so those comparisons are fair.

**Run it in Colab** with a GPU (Runtime → Change runtime type → T4 GPU). It takes about 10 minutes.

**Content note:** the datasets contain slurs. Nothing in this notebook prints a tweet.

In [1]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run("pip install -q 'scikit-learn>=1.6' && "
                   "git clone -q https://github.com/SanjanaJairam/abusive-language-detection.git && "
                   "cd abusive-language-detection && bash get_data.sh", shell=True, check=True)
    REPO = "abusive-language-detection"
else:
    REPO = ".."  # running from notebooks/ in a local copy
sys.path.insert(0, f"{REPO}/src")

In [2]:
import json, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from transformers import pipeline
from ald import audits, checks, data, dialect
from ald.config import CLASSES, HATE, NEITHER, OFFENSIVE, RESULTS
from ald.metrics import evaluate
from ald.pipeline import dialect_groups

try:
    import torch
    DEVICE = 0 if torch.cuda.is_available() else -1
except ImportError:
    DEVICE = -1
print("Using", "GPU" if DEVICE == 0 else "CPU (slow: switch to a GPU runtime)")

Using GPU


## Load the two models

Cardiff NLP's Twitter models expect usernames as `@user` and links as `http`, so tweets are normalised that way first.

In [3]:
hate_model = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-hate-latest", device=DEVICE, top_k=None)
off_model = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-offensive", device=DEVICE, top_k=None)

def normalise(text):
    text = re.sub(r"@\w+", "@user", str(text))
    return re.sub(r"https?://\S+|www\.\S+", "http", text)

def positive_score(outputs):
    """Score of the positive label (HATE / offensive), whatever the model calls it."""
    for o in outputs:
        name = o["label"].lower()
        if not (name.startswith("not") or name.startswith("non") or name == "label_0"):
            return o["score"]
    return 0.0

_cache = {}
def scores(texts):
    """[n, 2] array: P(hate) and P(offensive), cached so each text is scored once."""
    todo = [t for t in dict.fromkeys(texts) if t not in _cache]
    if todo:
        clean = [normalise(t) for t in todo]
        hs = hate_model(clean, batch_size=64, truncation=True, max_length=128)
        os_ = off_model(clean, batch_size=64, truncation=True, max_length=128)
        for t, h, o in zip(todo, hs, os_):
            _cache[t] = (positive_score(h), positive_score(o))
    return np.array([_cache[t] for t in texts])

def proba(texts):
    """Three-way probabilities: hate, offensive-but-not-hate, neither."""
    s = scores(list(texts))
    p_hate, p_off = s[:, 0], s[:, 1]
    return np.column_stack([p_hate, (1 - p_hate) * p_off, (1 - p_hate) * (1 - p_off)])

def labels(texts):
    s = scores(list(texts))
    return np.where(s[:, 0] >= 0.5, HATE, np.where(s[:, 1] >= 0.5, OFFENSIVE, NEITHER))

labels(["Had a great day at the beach with my family", "this damn printer jammed again"])

config.json:   0%|          | 0.00/888 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

...

model.safetensors: downloading bytes:           |  0.00B            

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

array([2, 1])

## Davidson test set (seen in training)

In [4]:
df = data.load_davidson()
train, val, test = data.split(df)
m = evaluate(test.label, labels(test.tweet))
ours = json.loads((Path(REPO) / "results" / "results.json").read_text())
final = ours["variants"]["+ identity swaps (final)"]["davidson_test"]
pd.DataFrame({
    "Modern model (has seen these tweets)": [m["accuracy"], m["macro_f1"], m["per_class"]["hate"]["recall"], m["per_class"]["hate"]["precision"]],
    "Final model from notebook 2": [final["accuracy"], final["macro_f1"], final["per_class"]["hate"]["recall"], final["per_class"]["hate"]["precision"]],
}, index=["accuracy", "macro F1", "hate recall", "hate precision"]).style.format("{:.1%}")

,Modern model (has seen these tweets),Final model from notebook 2
accuracy,39.0%,89.1%
macro F1,45.3%,73.3%
hate recall,88.5%,40.9%
hate precision,8.6%,39.3%


## HateCheck and identity terms (new to both models)

In [5]:
hc, hc_func = audits.hatecheck(labels)
idt, idt_terms = audits.identity_terms(labels)
pd.DataFrame({
    "Modern model": [hc["hate_detected"], hc["benign_wrongly_hate"], idt["overall_false_flag_rate"]],
    "Final model from notebook 2": [ours["hatecheck"]["final"]["hate_detected"], ours["hatecheck"]["final"]["benign_wrongly_hate"],
                                    ours["identity_terms"]["final"]["overall_false_flag_rate"]],
}, index=["HateCheck: hateful cases caught as hate", "HateCheck: harmless cases called hate", "Identity sentences flagged"]).style.format("{:.0%}")

,Modern model,Final model from notebook 2
HateCheck: hateful cases caught as hate,62%,14%
HateCheck: harmless cases called hate,14%,8%
Identity sentences flagged,16%,24%


Where the modern model goes wrong on HateCheck, and which identity terms it flags most in harmless sentences:

In [6]:
worst = hc_func.head(8)
worst.assign(kind=np.where(worst.gold_hateful, "hateful", "not hateful"))[["functionality", "kind", "n", "accuracy"]] \
     .style.format({"accuracy": "{:.0%}"}).hide(axis="index")

functionality,kind,n,accuracy
spell_space_add_h,hateful,173,34%
spell_char_swap_h,hateful,133,38%
negate_pos_h,hateful,140,41%
derog_impl_h,hateful,140,43%
spell_leet_h,hateful,173,49%
spell_char_del_h,hateful,140,51%
slur_homonym_nh,not hateful,30,57%
derog_neg_emote_h,hateful,140,64%


In [7]:
idt_terms.head(10)[["false_flag_rate"]].style.format("{:.0%}")

,false_flag_rate
term,
lesbian,100%
gay,99%
homosexual,97%
latinx,92%
queer,80%
taoist,65%
bisexual,61%
old,37%
sikh,22%


## Behavioural checks

In [8]:
c = checks.run_all(labels, proba)
f = ours["variants"]["+ identity swaps (final)"]["checks"]
rows = [("everyday tweets not flagged", "everyday_speech", "olid_pass_rate"),
        ("everyday examples not flagged", "everyday_speech", "examples_pass_rate"),
        ("negation not called hate", "negation", "not_called_hate_pass_rate"),
        ("same label across groups", "identity_swap", "consistent_pass_rate"),
        ("counter-speech not called hate", "counter_speech", "not_called_hate_pass_rate")]
pd.DataFrame({"Modern model": [c[g][k] for _, g, k in rows], "Final model from notebook 2": [f[g][k] for _, g, k in rows]},
             index=[r[0] for r in rows]).style.format("{:.0%}")

,Modern model,Final model from notebook 2
everyday tweets not flagged,93%,91%
everyday examples not flagged,87%,87%
negation not called hate,77%,94%
same label across groups,30%,39%
counter-speech not called hate,78%,88%


"Everyday tweets" come from OLID's test set; the offensive model was trained on OLID's training set, so this check also favours it.

## Dialect audit (seen in training)

Same design as notebook 2: TwitterAAE estimates how much of each tweet's language is African American English, and a tweet counts as AAE- or White-aligned above 60%. The modern model is pretrained, so it predicts every tweet directly.

In [9]:
props = np.array([p if (p := dialect.proportions(t)) is not None else [np.nan] * 4 for t in df.tweet])
groups = dialect_groups(props, 0.6)
d = audits.dialect_disparity(df.tweet, df.label, labels(df.tweet), groups)
fd = ours["dialect"]["threshold_0.6"]["Final model"]
pd.DataFrame({
    "Modern model (has seen these tweets)": [d["aae"]["false_flag_rate"], d["white"]["false_flag_rate"], d["false_flag_ratio"]],
    "Final model from notebook 2": [fd["aae"]["false_flag_rate"], fd["white"]["false_flag_rate"], fd["false_flag_ratio"]],
}, index=["harmless AAE-aligned flagged", "harmless White-aligned flagged", "ratio"]).T \
  .style.format({"harmless AAE-aligned flagged": "{:.0%}", "harmless White-aligned flagged": "{:.0%}", "ratio": "{:.1f}×"})

,harmless AAE-aligned flagged,harmless White-aligned flagged,ratio
Modern model (has seen these tweets),44%,20%,2.2×
Final model from notebook 2,37%,6%,6.5×


In [10]:
audits.false_flag_by_aae_share(df.label, labels(df.tweet), props[:, 0]) \
      .rename(columns={"bin": "estimated AAE share", "n": "harmless tweets", "false_flag_rate": "flagged"}) \
      .style.format({"flagged": "{:.0%}"}).hide(axis="index")

estimated AAE share,harmless tweets,flagged
"(-0.001, 0.2]",2463,20%
"(0.2, 0.4]",1263,33%
"(0.4, 0.6]",346,43%
"(0.6, 0.8]",84,45%
"(0.8, 1.0]",6,33%


## Save the results

Everything above goes into `results/modern_model.json`. In Colab, the last cell downloads it so it can be added to the repository.

In [11]:
def to_json(o):
    if isinstance(o, dict): return {str(k): to_json(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_json(v) for v in o]
    if isinstance(o, (np.floating, float)): return None if np.isnan(o) else round(float(o), 4)
    if isinstance(o, np.integer): return int(o)
    return o

out = {"models": ["cardiffnlp/twitter-roberta-base-hate-latest", "cardiffnlp/twitter-roberta-base-offensive"],
       "note": "The hate model was trained on data including Davidson et al. (2017) and OLID; Davidson and dialect results are in-sample.",
       "davidson_test": m, "hatecheck": hc, "identity_terms": idt, "checks": c,
       "dialect_threshold_0.6": d,
       "hatecheck_by_functionality": hc_func.to_dict("records"),
       "identity_false_flags": idt_terms.reset_index().to_dict("records")}
path = Path(REPO) / "results" / "modern_model.json"
path.write_text(json.dumps(to_json(out), indent=2))
print("Saved", path)
if IN_COLAB:
    from google.colab import files
    files.download(str(path))

Saved abusive-language-detection/results/modern_model.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What to look for

- **HateCheck and identity terms** are the fair comparison. If the modern model still calls harmless identity sentences or counter-speech hate, the problem isn't just old methods.
- **The dialect gap** is measured on tweets the modern model has seen, so even a small gap there is telling: it learned the annotators' pattern along with the labels.

**References.** Antypas, D. & Camacho-Collados, J. (2023). Robust hate speech detection in social media: A cross-dataset empirical evaluation. *WOAH*. · Barbieri, F., et al. (2020). TweetEval. *Findings of EMNLP*. · Plus the datasets and audits credited in the README.